# 03 — Reddit NormVio Collection

This notebook replaces live Reddit API collection with the **NormVio** research dataset.
The notebook loads the public redacted NormVio JSONL files from a local folder, standardizes the Reddit comment IDs, conversation IDs, subreddit names, rule texts, violation categories, derailing labels, restored status, and redacted context/comment fields. The cleaned data is stored in the same SQLite database as the Bluesky and Lemmy data.


In [ ]:
%pip install pandas

In [46]:
from pathlib import Path
import sqlite3
import json
import pandas as pd
from datetime import datetime, timezone

DB_PATH = Path("../data/moderation.db")
NORMVIO_DIR = Path("../data/raw/normvio")

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
NORMVIO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Database path : {DB_PATH.resolve()}")
print(f"NormVio folder: {NORMVIO_DIR.resolve()}")

Database path : C:\Users\iRemont\content-moderation\data\moderation.db
NormVio folder: C:\Users\iRemont\content-moderation\data\raw\normvio


In [47]:
jsonl_files = sorted(NORMVIO_DIR.glob("*.jsonl"))

print(f"Looking in: {NORMVIO_DIR.resolve()}")
print(f"Found {len(jsonl_files)} JSONL files:")

for file in jsonl_files:
    print(" -", file.name)

if not jsonl_files:
    raise FileNotFoundError(
        f"No .jsonl files found in {NORMVIO_DIR.resolve()}. "
    )

Looking in: C:\Users\iRemont\content-moderation\data\raw\normvio
Found 3 JSONL files:
 - dev.jsonl
 - test.jsonl
 - train.jsonl


In [32]:
dfs = []

for file in jsonl_files:
    print(f"Loading {file.name}...")
    df = pd.read_json(file, lines=True)
    df["source_file"] = file.name
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(raw_df):,} total rows.")
raw_df.head()

Loading dev.jsonl...
Loading test.jsonl...
Loading train.jsonl...
Loaded 52,012 total rows.


,comment_id,conv_id,subreddit,bool_derail,rule_texts,cats,is_restored,redacted_context,redacted_final_comment,source_file
0,gv2uq3h,mtkwwy~gv1mmbj_mod,ANormalDayInRussia,True,Keep it civil. Don't be cyka.,incivility,1.0,"[{'id': 'mtkwwy~gv1mmbj_mod'}, {'id': 'gv0bwrp...",{'id': 'gv1mmbj~gv1mmbj'},dev.jsonl
1,fqh7ks2,gikv2w~fqg6rmy_mod,youngpeopleyoutube,True,Censor the kid's username (if a channel) and p...,"format,doxxing",1.0,"[{'id': 'gikv2w~fqg6rmy_mod'}, {'id': 'fqfel1a...",{'id': 'fqg6rmy~fqg6rmy'},dev.jsonl
2,guvkmvq,msc1n4~gurvk6l_mod,halo,True,Show basic courtesy and respect,incivility,1.0,"[{'id': 'msc1n4~gurvk6l_mod'}, {'id': 'guroriv...",{'id': 'gurvk6l~gurvk6l'},dev.jsonl
3,gbx30fo,jhmmp6~ga01brb_mod,Doom,True,Don’t be a dick.,incivility,1.0,[{'id': 'jhmmp6~ga01brb_mod'}],{'id': 'ga01brb~ga01brb'},dev.jsonl
4,gr1m5z5,m47zp0~gqsz74r_mod,universe,True,Post titles must contain a universe-related su...,"format,off-topic",1.0,[{'id': 'm47zp0~gqsz74r_mod'}],{'id': 'gqsz74r~gqsz74r'},dev.jsonl


In [33]:
print("Columns:")
for col in raw_df.columns:
    print("-", col)

raw_df.head()

Columns:
- comment_id
- conv_id
- subreddit
- bool_derail
- rule_texts
- cats
- is_restored
- redacted_context
- redacted_final_comment
- source_file


,comment_id,conv_id,subreddit,bool_derail,rule_texts,cats,is_restored,redacted_context,redacted_final_comment,source_file
0,gv2uq3h,mtkwwy~gv1mmbj_mod,ANormalDayInRussia,True,Keep it civil. Don't be cyka.,incivility,1.0,"[{'id': 'mtkwwy~gv1mmbj_mod'}, {'id': 'gv0bwrp...",{'id': 'gv1mmbj~gv1mmbj'},dev.jsonl
1,fqh7ks2,gikv2w~fqg6rmy_mod,youngpeopleyoutube,True,Censor the kid's username (if a channel) and p...,"format,doxxing",1.0,"[{'id': 'gikv2w~fqg6rmy_mod'}, {'id': 'fqfel1a...",{'id': 'fqg6rmy~fqg6rmy'},dev.jsonl
2,guvkmvq,msc1n4~gurvk6l_mod,halo,True,Show basic courtesy and respect,incivility,1.0,"[{'id': 'msc1n4~gurvk6l_mod'}, {'id': 'guroriv...",{'id': 'gurvk6l~gurvk6l'},dev.jsonl
3,gbx30fo,jhmmp6~ga01brb_mod,Doom,True,Don’t be a dick.,incivility,1.0,[{'id': 'jhmmp6~ga01brb_mod'}],{'id': 'ga01brb~ga01brb'},dev.jsonl
4,gr1m5z5,m47zp0~gqsz74r_mod,universe,True,Post titles must contain a universe-related su...,"format,off-topic",1.0,[{'id': 'm47zp0~gqsz74r_mod'}],{'id': 'gqsz74r~gqsz74r'},dev.jsonl


In [34]:
print("Shape:", raw_df.shape)
print("\nMissing values:")
print(raw_df.isna().sum().sort_values(ascending=False).head(20))

Shape: (52012, 10)

Missing values:
is_restored               31875
comment_id                    0
conv_id                       0
subreddit                     0
rule_texts                    0
bool_derail                   0
cats                          0
redacted_context              0
redacted_final_comment        0
source_file                   0
dtype: int64


In [35]:
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

In [36]:
import json

raw_df_sql = raw_df.copy()

for col in raw_df_sql.columns:
    raw_df_sql[col] = raw_df_sql[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x
    )

conn = get_conn()

raw_df_sql.to_sql(
    "reddit_normvio_raw",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Saved raw NormVio data to table: reddit_normvio_raw")

Saved raw NormVio data to table: reddit_normvio_raw


In [37]:
conn = get_conn()

count = conn.execute("SELECT COUNT(*) FROM reddit_normvio_raw").fetchone()[0]

conn.close()

print(f"Rows saved in reddit_normvio_raw: {count:,}")

Rows saved in reddit_normvio_raw: 52,012


In [38]:
reddit_clean = pd.DataFrame(index=raw_df.index)

reddit_clean["platform"] = "reddit"
reddit_clean["source_dataset"] = "NormVio"

reddit_clean["reddit_id"] = raw_df["comment_id"]
reddit_clean["conversation_id"] = raw_df["conv_id"]
reddit_clean["subreddit"] = raw_df["subreddit"]

reddit_clean["post_type"] = "comment"

reddit_clean["rule_texts"] = raw_df["rule_texts"]
reddit_clean["violation_category"] = raw_df["cats"]

reddit_clean["is_derailing"] = raw_df["bool_derail"]
reddit_clean["is_restored"] = raw_df["is_restored"]

reddit_clean["redacted_context"] = raw_df["redacted_context"]
reddit_clean["redacted_text"] = raw_df["redacted_final_comment"]

reddit_clean["moderation_signal"] = "norm_violation"
reddit_clean["source_file"] = raw_df["source_file"]

reddit_clean["collected_at"] = datetime.now(timezone.utc).isoformat()

reddit_clean.head()

,platform,source_dataset,reddit_id,conversation_id,subreddit,post_type,rule_texts,violation_category,is_derailing,is_restored,redacted_context,redacted_text,moderation_signal,source_file,collected_at
0,reddit,NormVio,gv2uq3h,mtkwwy~gv1mmbj_mod,ANormalDayInRussia,comment,Keep it civil. Don't be cyka.,incivility,True,1.0,"[{'id': 'mtkwwy~gv1mmbj_mod'}, {'id': 'gv0bwrp...",{'id': 'gv1mmbj~gv1mmbj'},norm_violation,dev.jsonl,2026-06-06T03:35:48.043892+00:00
1,reddit,NormVio,fqh7ks2,gikv2w~fqg6rmy_mod,youngpeopleyoutube,comment,Censor the kid's username (if a channel) and p...,"format,doxxing",True,1.0,"[{'id': 'gikv2w~fqg6rmy_mod'}, {'id': 'fqfel1a...",{'id': 'fqg6rmy~fqg6rmy'},norm_violation,dev.jsonl,2026-06-06T03:35:48.043892+00:00
2,reddit,NormVio,guvkmvq,msc1n4~gurvk6l_mod,halo,comment,Show basic courtesy and respect,incivility,True,1.0,"[{'id': 'msc1n4~gurvk6l_mod'}, {'id': 'guroriv...",{'id': 'gurvk6l~gurvk6l'},norm_violation,dev.jsonl,2026-06-06T03:35:48.043892+00:00
3,reddit,NormVio,gbx30fo,jhmmp6~ga01brb_mod,Doom,comment,Don’t be a dick.,incivility,True,1.0,[{'id': 'jhmmp6~ga01brb_mod'}],{'id': 'ga01brb~ga01brb'},norm_violation,dev.jsonl,2026-06-06T03:35:48.043892+00:00
4,reddit,NormVio,gr1m5z5,m47zp0~gqsz74r_mod,universe,comment,Post titles must contain a universe-related su...,"format,off-topic",True,1.0,[{'id': 'm47zp0~gqsz74r_mod'}],{'id': 'gqsz74r~gqsz74r'},norm_violation,dev.jsonl,2026-06-06T03:35:48.043892+00:00


In [39]:
reddit_clean_sql = reddit_clean.copy()

for col in reddit_clean_sql.columns:
    reddit_clean_sql[col] = reddit_clean_sql[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x
    )

In [40]:
conn = get_conn()

reddit_clean_sql.to_sql(
    "reddit_normvio_clean",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Saved cleaned NormVio data to table: reddit_normvio_clean")

Saved cleaned NormVio data to table: reddit_normvio_clean


In [48]:
conn = get_conn()

validation = pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT reddit_id) AS unique_comments,
        COUNT(DISTINCT subreddit) AS unique_subreddits,
        SUM(CASE WHEN platform IS NULL THEN 1 ELSE 0 END) AS missing_platform,
        SUM(CASE WHEN source_dataset IS NULL THEN 1 ELSE 0 END) AS missing_source_dataset
    FROM reddit_normvio_clean
""", conn)

conn.close()

validation

,total_rows,unique_comments,unique_subreddits,missing_platform,missing_source_dataset
0,52012,20137,2310,0,0


In [49]:
conn = get_conn()

count = conn.execute("SELECT COUNT(*) FROM reddit_normvio_clean").fetchone()[0]

conn.close()

print(f"Rows saved in reddit_normvio_clean: {count:,}")

Rows saved in reddit_normvio_clean: 52,012


In [50]:
conn = get_conn()

df_subreddits = pd.read_sql_query("""
    SELECT subreddit, COUNT(*) AS count
    FROM reddit_normvio_clean
    GROUP BY subreddit
    ORDER BY count DESC
    LIMIT 20
""", conn)

conn.close()

df_subreddits

,subreddit,count
0,Coronavirus,2104
1,AmItheAsshole,1630
2,classicwow,1082
3,CanadaPolitics,1068
4,Games,857
5,RPClipsGTA,752
6,heroesofthestorm,642
7,LabourUK,625
8,ShingekiNoKyojin,595
9,MakeMyCoffin,551


In [51]:
conn = get_conn()

df_categories = pd.read_sql_query("""
    SELECT violation_category, COUNT(*) AS count
    FROM reddit_normvio_clean
    GROUP BY violation_category
    ORDER BY count DESC
    LIMIT 20
""", conn)

conn.close()

df_categories

,violation_category,count
0,incivility,24104
1,spam,4874
2,harassment,4621
3,off-topic,3177
4,content,2883
5,format,2563
6,"content,format",1160
7,trolling,1045
8,meta-rules,1031
9,hatespeech,994


In [52]:
conn = get_conn()

df_sample = pd.read_sql_query("""
    SELECT 
        reddit_id,
        subreddit,
        violation_category,
        rule_texts,
        is_derailing,
        is_restored,
        redacted_text
    FROM reddit_normvio_clean
    LIMIT 10
""", conn)

conn.close()

df_sample

,reddit_id,subreddit,violation_category,rule_texts,is_derailing,is_restored,redacted_text
0,gv2uq3h,ANormalDayInRussia,incivility,Keep it civil. Don't be cyka.,1,1.0,"{""id"": ""gv1mmbj~gv1mmbj""}"
1,fqh7ks2,youngpeopleyoutube,"format,doxxing",Censor the kid's username (if a channel) and p...,1,1.0,"{""id"": ""fqg6rmy~fqg6rmy""}"
2,guvkmvq,halo,incivility,Show basic courtesy and respect,1,1.0,"{""id"": ""gurvk6l~gurvk6l""}"
3,gbx30fo,Doom,incivility,Don’t be a dick.,1,1.0,"{""id"": ""ga01brb~ga01brb""}"
4,gr1m5z5,universe,"format,off-topic",Post titles must contain a universe-related su...,1,1.0,"{""id"": ""gqsz74r~gqsz74r""}"
5,e8u31pz,StreetFighter,incivility,The Civility Rule,1,1.0,"{""id"": ""e8u12kt~e8u12kt""}"
6,gtc1oee,loseit,spam,No Self Promotion,1,1.0,"{""id"": ""gtbce2z~gtbce2z""}"
7,gftjeyq,BokuNoShipAcademia,incivility,Be Respectful of All Ships and Shippers,1,1.0,"{""id"": ""gfszyuz~gfszyuz""}"
8,ftl6o9c,TexasPolitics,incivility,Be Civil and Make an Effort,1,1.0,"{""id"": ""ftify3k~ftify3k""}"
9,dheqsp4,sushi,incivility,"It's okay to have a different opinion, it's no...",1,1.0,"{""id"": ""dh78s7h~dh78s7h""}"


In [53]:
conn = get_conn()

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""", conn)

conn.close()

tables

,name
0,bsky_labels
1,bsky_posts
2,label_definitions
3,labelers
4,lemmy_modlog
5,lemmy_posts
6,reddit_normvio_clean
7,reddit_normvio_raw
8,sqlite_sequence


## Summary

We store the Reddit NormVio data in the same SQLite database, `moderation.db`, as the Lemmy and Bluesky data. This keeps all platform data in one place and makes cross-platform analysis easier.

The Reddit table differs from the Lemmy and Bluesky tables because NormVio is a redacted research dataset. Instead of live Reddit post text, it provides Reddit comment IDs, conversation IDs, subreddits, rule texts, violation categories, derailing labels, restored status, and redacted context/comment identifiers.